---
title: "The Federal Laws Corpus: Mexico's Statute Book, One Snapshot per Reform"
subtitle: "How 3,724 dated versions of 315 federal instruments were assembled, and what can be trusted about them"
date: 2026-08-28
abstract: >
  Mexico's federal legislation is easy to read today and hard to read
  historically. The Chamber of Deputies publishes the text of every federal law
  as it stands right now, together with the list of decrees that produced it,
  but not the intermediate wording those decrees left behind; the Diario
  Oficial de la Federación publishes each reform decree and nothing else, so
  the consolidated text of a law on a given date exists only as the sum of
  everything published before it. This page documents a corpus that closes that
  gap: 3,724 Markdown snapshots covering 315 of the 316 federal instruments in
  the Chamber's catalogue, each one the complete text of an instrument as it
  read on the day a particular reform took effect, taken from the Supreme
  Court's legislative database, and — for 87.7% of them — tied to the specific
  Diario Oficial provision that caused the change. We describe where the
  material comes from, the two independent signals used to identify that
  provision, the four catalogue entries that no unattended crawl could handle,
  and how the collection is published and can be downloaded.
---


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INGEOTEC/LegalIA/blob/master/website/pages/leyes.ipynb)


In [ ]:
#| label: setup
#| code-summary: "Imports, the styling shared by every figure, and the aggregate summary this page reads"
import json
from pathlib import Path
from urllib.request import urlopen

import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown

# Validated categorical palette (light mode), in fixed assignment order — the
# same one `DOF Titles` and `Dataset` use, so a colour means the same thing
# across the site.
BLUE, GREEN, MAGENTA, YELLOW = "#2a78d6", "#008300", "#e87ba4", "#eda100"
LIGHT_BLUE = "#9ec5f4"  # step 200 of the blue sequential ramp
INK, MUTED, GRID, AXIS = "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

PLOTLY_FONT = "system-ui, -apple-system, 'Segoe UI', Helvetica, Arial, sans-serif"


def style_plotly(fig, *, yaxis_title="", height=430):
    fig.update_layout(
        font=dict(family=PLOTLY_FONT, size=13, color=INK),
        paper_bgcolor="white",
        plot_bgcolor="white",
        height=height,
        margin=dict(l=64, r=24, t=16, b=40),
        hoverlabel=dict(bgcolor="white", bordercolor=GRID, font=dict(color=INK, size=12)),
        legend=dict(font=dict(size=12, color=INK)),
    )
    fig.update_xaxes(
        showgrid=False, showline=False, zeroline=False,
        ticks="", tickfont=dict(color=MUTED), title_text="",
    )
    fig.update_yaxes(
        showgrid=True, gridcolor=GRID, gridwidth=1,
        showline=False, zeroline=False,
        ticks="", tickfont=dict(color=MUTED),
        title_text=yaxis_title, title_font=dict(color=INK, size=13),
    )
    return fig


PLOTLY_CONFIG = {
    "displaylogo": False,
    "displayModeBar": "hover",
    "modeBarButtonsToRemove": ["lasso2d", "select2d", "autoScale2d"],
}

# Every figure and every number in the prose below comes from this one file.
# The corpus itself lives in the `scjn-leyes` release — one tarball per
# instrument — and is never committed to git, so, unlike `Dataset`, which
# queries the GitHub API live, this page reads the aggregate summary
# `scripts/resume_scjn_leyes.py` distils from the crawl the release was
# packaged from: counts, per-year totals and one row per instrument,
# regenerated by re-running that script and versioned alongside the page.
RESUMEN = "data/scjn-leyes-summary.json"
RAW = "https://raw.githubusercontent.com/INGEOTEC/LegalIA/master/website/pages/" + RESUMEN
local = Path(RESUMEN)
if local.is_file():  # rendering the site
    resumen = json.loads(local.read_text(encoding="utf-8"))
else:               # running the notebook on Colab
    with urlopen(RAW) as respuesta:
        resumen = json.loads(respuesta.read().decode("utf-8"))

instrumentos = pd.DataFrame(resumen["instruments"])
por_anio = pd.Series(
    {int(anio): n for anio, n in resumen["snapshots_per_year"].items()}
).sort_index()

# Headline figures, named once here and reused by the prose, the table and the
# figure captions so no number on this page is typed by hand twice.
SNAPSHOTS = resumen["snapshots"]
CATALOGO = resumen["catalogue_entries"]
CON_DIRECTORIO = resumen["instruments_with_directory"]
ESTADOS = resumen["title_link_status"]
POR_CONFIRMACION = resumen["title_link_status_by_confirmation"]
ENLAZADOS = resumen["linked_by_title"]
CONFIRMADOS = resumen["content_diff_confirmed"]
# A date is identified when the title signal resolved it *or* when the content
# diff resolved a date the title signal left ambiguous — the two signals are
# complementary, and this is the coverage figure that matters to a reader.
RESCATADOS = POR_CONFIRMACION.get("ambiguous|confirmed", 0)
IDENTIFICADOS = ENLAZADOS + RESCATADOS


In [ ]:
# On Colab, install the packages this notebook needs:
# %pip install pandas plotly


## What a snapshot is

The unit of this collection is not a law and not a decree: it is a **snapshot**
— one plain-text file holding the *complete* text of one federal instrument as
it read on one particular date. The file is named after that date
(`01-04-2025.md`) and lives in a directory named after the instrument
(`cpeum/` for the Constitution, `lft/` for the Federal Labour Law), so an
instrument's directory is its own history laid out in files: as many files as
there are dates on which its wording changed.

This matters because it is the version of the statute book that is otherwise
missing. To know what the Federal Labour Law said in 1998 one can consult the
1998 reform decree in the *Diario Oficial* — but a decree only says which
articles were replaced and by what, not what the resulting law read like as a
whole. Reconstructing the whole from the parts means replaying every decree in
order, which the [`nota2md`](https://pypi.org/project/nota2md/) package can do
(`reconstruct_legal_provisions`) but which depends on every intermediate decree
being available as digital text — and for anything published before roughly
1999 it is not: it exists only as scanned images that have to be read by OCR.
A snapshot sidesteps the reconstruction entirely, because somebody has already
done the consolidation: it *is* the law as of that date, in one piece.

The format is Markdown, and that choice is deliberate. Markdown is plain text,
so the files are readable in any editor, searchable with `grep`, diffable line
by line against each other, and stable in version control — the same reasons
the rest of LegalIA converts legal texts to Markdown rather than keeping PDFs
or `.docx` files around. It also keeps just enough structure to be useful — a
heading is a heading, an emphasized run stays emphasized — without pretending
to a legal-document schema the source never had. Each file opens with a short
YAML header recording where it came from, which is described in detail
below; everything after the header is the text of the law.


## Why the Supreme Court and not only the Chamber of Deputies

The obvious place to look for federal legislation is the Chamber of Deputies'
own [LeyesBiblio](https://www.diputados.gob.mx/LeyesBiblio/index.htm), and
LegalIA does use it — but for a different purpose than one might expect. What
LeyesBiblio offers is the **current** text of each law, plus the list of
reform decrees that produced it. That is a catalogue and a bibliography: it
tells you which instruments exist and which *Diario Oficial* publications
changed them, but not what any of them said in between.

The Supreme Court's legislative database
([legislacion.scjn.gob.mx](https://legislacion.scjn.gob.mx/Buscador/)) is
organized the other way around. For each instrument it keeps a table of
reforms, and for each row of that table it serves the consolidated text of the
instrument **as it stood after that reform** — a full document, not a summary
of the change. The Court maintains this because its own work requires it: a
case is decided under the law in force at the relevant time, not under the law
in force today. That editorial effort, undertaken for judicial reasons, is
exactly what a historical corpus needs, and it is why the Court is this
collection's primary source of *content* while the Chamber remains its source
of *scope* — which instruments exist, what each is called, and when each was
last reformed.

### A caveat about the source

The Court's database is a source of convenience, not the source of authority.
The authoritative text of Mexican federal law is what the *Diario Oficial de la
Federación* published; the Court's consolidated versions are an editorial
product built on top of it, and they behave like one:

- **There is no stable address for a document.** The search interface hands out
  URLs whose `?q=` token belongs to the session that generated it, so a link
  cannot be cited, bookmarked, or re-fetched later. The only reproducible
  record of how a file was obtained is what was searched for and what came
  back — which is precisely why every snapshot carries that pair in its header.
- **The search can return the wrong document entirely.** A query for one law
  can rank a different law, or an internal court agreement that merely
  mentions the law, above the law itself. Two mechanisms guard against this: a
  similarity score between what was searched and what was found, recorded in
  every file, and a filter that discards the Court's own internal agreements.
  Neither is infallible, which is why publication of this collection is a
  manual act, described at the end of this page.
- **Editorial matter is mixed into the text.** The Court annotates its
  consolidated texts with editor's notes ("N. DE E.") explaining, for example,
  that a reform was later invalidated. These are the Court speaking, not the
  legislature, and are stripped at conversion time so they never masquerade as
  statutory text.

Because the source is editorial, every file declares its provenance in its own
header — `fuente: scjn` — and nothing in LegalIA ever silently mixes these
texts with Markdown built from the *Diario Oficial* itself.


## From two catalogues to one corpus

The pipeline that builds the collection has four stages, run in order, each
one a separate script under
[`scripts/`](https://github.com/INGEOTEC/LegalIA/tree/master/scripts)
([@fig-pipeline]). What follows describes not only what each stage does but
what it *decides*, since the decisions are where a corpus of this kind either
earns or loses a reader's trust.


::: {#fig-pipeline}

```{mermaid}
flowchart TD
    D[Cámara de Diputados<br/>LeyesBiblio] -->|nombre, abrev, actualizado| C[catalogo.json<br/>extract_scjn_titles.py]
    C -->|search by nombre| S[SCJN buscador<br/>fetch_scjn_legislacion.py]
    S -->|one .docx per reform| M[Markdown snapshots<br/>DD-MM-YYYY.md + YAML header]
    A[notas-archivo release] -->|codNota + titulo + fecha| T[titles stream<br/>legal_provisions_titles]
    M --> L[enlaza_scjn_legislacion.py]
    T --> L
    N[dofjson.get_nota<br/>DOF Markdown] --> L
    L -->|indice.json per instrument| P[empaqueta_scjn_leyes.py<br/>one slug.tgz + MANIFEST.md]
    P -.->|manual review, manual upload| R[(scjn-leyes release)]
```

The pipeline. Two catalogues enter — the Chamber of Deputies' list of
instruments and the DOF's own archive of published provision titles — and one
corpus of dated snapshots, each linked to the provision that produced it,
comes out. The dotted arrow is the only step no script performs.

:::


### Seeding: which instruments exist

The first stage asks the Chamber of Deputies one question — what is in the
catalogue? — and keeps three fields per entry: the instrument's official
`nombre`, its short slug (`abrev`, which becomes the directory name), and
`actualizado`, the date of its most recent reform. The catalogue's own list of
reform provisions is deliberately **not** carried forward. That is a design
correction, not an oversight: an earlier design linked snapshots to *Diario
Oficial* provisions by trusting the Chamber's list of them, which makes the
corpus inherit whatever gaps and errors that list has. The current design
derives the link independently, from the DOF's own archive, and uses the
Chamber only for scope. Where the two disagree, the disagreement is visible
instead of absorbed.

`actualizado` earns its place by making refreshes cheap. Once the whole
collection has been crawled start to finish, a later run can skip an
instrument that has not been reformed since — but only if it already has
snapshots on disk. An instrument with no snapshots is retried on every single
refresh, forever, which is what makes the collection self-healing for laws the
Court has not indexed yet: when the Court catches up, the next refresh picks
the law up with nothing to configure by hand.

### Crawling: finding the right document

The second stage searches the Court's database for each catalogue name, and
this is where most of the difficulty lives. A search returns a ranked list of
documents; the crawler has to decide which one, if any, is the instrument it
was looking for. It discards the Court's own internal agreements, rejects
candidates whose title is too far from what was searched for, flags as
`sospechoso` (*suspect*) the ones that are close but not close enough, and
picks the best remaining match. For the winner it walks the reform table,
downloads one `.docx` per row, converts it to Markdown and strips the
editorial notes.

One limit of this stage is worth stating plainly, because it is live rather
than historical. The results grid paginates, and the crawler reads a single
page — the largest the page-size dropdown offers, **50 records**. An instrument
mentioned by more than 50 documents would therefore be invisible even though
the Court holds it, and the failure looks like an ordinary absence rather than
an error. That this is a real failure mode and not a hypothetical one is known
from the Income Tax Law: searching for it returns 42 records, the top ten of
which are internal agreements of the Court's own plenary session that mention
the law in passing, and while the crawler read only the first ten the statute
itself sat at position 14, invisible. Reading 50 at once fixed that case and
every other one in the catalogue — no entry in it exceeds 50 today — but the
trade-off was taken knowingly, and the number is recorded here so that a future
gap of this shape is recognized immediately instead of rediscovered.

The stage is resumable at two levels — a file already on disk is never
re-downloaded, and the index of the last instrument attempted is checkpointed,
so an interrupted run resumes rather than restarts — and rate-limited, out of
courtesy to an unofficial service that was never designed to be crawled.

### The header as a provenance record

Every snapshot opens with a YAML header. It is short, and each line is there
to answer a question a sceptical reader would ask:

```yaml
---
fuente: scjn
nombre_buscado: CONSTITUCIÓN Política de los Estados Unidos Mexicanos
ordenamiento: CONSTITUCION POLITICA DE LOS ESTADOS UNIDOS MEXICANOS
fecha_publicacion: 01-04-2025
fecha_expedicion: 01-04-2025
categoria: DECRETO
ratio_similitud: 1.000
sospechoso: false
---
```

- **`fuente`** — where the text came from: `scjn` for the Court's
  consolidated version, `dof` for the handful of files taken directly from the
  *Diario Oficial* (see the cases below). This is the field that keeps an
  editorial text from ever being mistaken for the official one.
- **`ordenamiento`** — the title of the document the **Supreme Court**
  actually served, verbatim.
- **`nombre_buscado`** — the exact string that was searched for, which is the
  instrument's name as the **Chamber of Deputies' catalogue** gives it (the
  `nombre` seeded in the first stage) or, for the one entry that needs it, the
  `nombre_scjn` override described below. The pair is therefore not only *what
  was asked* / *what came back*: it is **one source's name for the law against
  the other source's name for the same law**, which is why the two can differ
  at all and why the similarity score below is worth recording. Because the
  Court's search has no citable URL, the pair is also the only way to reproduce
  by hand how a given file was reached. It is written only when the two differ
  after normalizing accents, case and whitespace — so its mere presence marks a
  file whose identification deserves a second look, and
  `grep -l nombre_buscado:` is a complete audit.
- **`ratio_similitud`** and **`sospechoso`** — how close those two strings are
  (1.000 meaning identical after normalization) and whether that closeness
  fell into the band where a match is kept but flagged. They record the
  crawler's confidence rather than hiding it behind a yes/no decision.
- **`fecha_publicacion`** and **`fecha_expedicion`** — the date the reform was
  published and the date it was issued, as the Court's reform table gives
  them; the first is the snapshot's identity and its file name.
- **`categoria`** — the kind of instrument that produced the reform, typically
  `DECRETO`.

The header plays the same role here that the raw JSON record of a note plays
on the `Dataset` page: it is the level at which a reader can check the data
rather than take it on faith.


### Linking each snapshot to the provision that caused it

The final stage of the pipeline answers a question the Court's database does
not: **which published provision produced this version of the law?** Its
output is one `indice.json` per instrument, pairing each snapshot with a
`codNota` — the *Diario Oficial*'s own identifier for a legal provision.

It is worth being explicit about why this link is worth the trouble, since it
is the most laborious part of the whole pipeline. Four things depend on it.

First, **verification**. A snapshot is an editorial product of the Court; the
provision it points to is the official text. Having both means any claim about
the corpus can be checked against the gazette — and, once digital text exists
on both sides, checked automatically rather than by hand.

Second, **joining**. The `codNota` is the key that the rest of LegalIA is
built around: it is what the DOF notes archive is indexed by, what the
[titles dataset](titles.ipynb) carries for all 1.2 million published
provisions, and what the reform-history release records. A linked snapshot
therefore inherits everything already known about its provision — its official
title, its issuing body, its full text where digitized — and lets this corpus
be used together with the others instead of beside them.

Third, **dating and ordering**. A snapshot's file name records a publication
date, but a date is not an identifier: the gazette can publish several reforms
to the same law on the same day, and an instrument's history is not
well-ordered until each version is tied to a specific provision.

Fourth, **provenance for anything built downstream**. Any dataset derived
from these snapshots — a diff of what each reform changed, a training corpus,
a citation graph — is only as citable as its link back to the official
record.

Two independent signals produce the link, and both are always computed,
because a weaker link is never what one actually wants.

**The title signal** looks at every provision the gazette published on the
snapshot's own date and keeps those whose official title explicitly names the
instrument. If exactly one candidate remains — after setting aside any
provision an earlier snapshot of the same instrument already claimed — the
date is linked. Each date therefore ends in one of four states: `linked` (one
candidate, resolved), `ambiguous` (several plausible candidates, unresolved),
`claimed` (the only candidate was already taken by another snapshot) or `none`
(no provision that day names the instrument at all, which is what one expects
for the years before the gazette's metadata was catalogued in detail).

**The content signal** attacks the `ambiguous` case, which the title alone
cannot resolve. If several provisions published the same day all name the law,
the one that actually caused *this* version is the one whose own text accounts
for what changed between this snapshot and the previous one. So the pipeline
takes the difference between consecutive snapshots, fetches each candidate
provision's own text from the gazette, and asks which candidate explains the
difference. This is a comparatively expensive computation — it needs the full
text of every candidate — and its intermediate results are not published: what
is kept is the verdict, the identifier of the confirming provision and a score,
not the diff itself.


## What the corpus looks like today

[@tbl-overview] gives the size of the collection as it stands. Every figure on
this page is computed from the summary file described in the methodological
note, not typed in by hand.


In [ ]:
#| label: tbl-overview
#| tbl-cap: "The federal laws corpus at a glance. Percentages of snapshots are over the 3,724 total."
filas = [
    ("Instruments in the Chamber of Deputies' catalogue", f"{CATALOGO:,}", ""),
    ("With a directory of snapshots", f"{CON_DIRECTORIO:,}",
     f"{CON_DIRECTORIO / CATALOGO * 100:.1f}% of the catalogue"),
    ("Markdown snapshots", f"{SNAPSHOTS:,}", ""),
    ("Snapshots taken from the SCJN", f"{resumen['snapshots_by_source'].get('scjn', 0):,}",
     f"{resumen['snapshots_by_source'].get('scjn', 0) / SNAPSHOTS * 100:.1f}%"),
    ("Snapshots taken from the DOF (hand-built cases)",
     f"{resumen['snapshots_by_source'].get('dof', 0):,}",
     f"{resumen['snapshots_by_source'].get('dof', 0) / SNAPSHOTS * 100:.2f}%"),
    ("Identified by the title signal alone", f"{ENLAZADOS:,}",
     f"{ENLAZADOS / SNAPSHOTS * 100:.1f}%"),
    ("Ambiguous dates resolved by the content signal", f"{RESCATADOS:,}",
     f"{RESCATADOS / SNAPSHOTS * 100:.1f}%"),
    ("Identified by either signal", f"{IDENTIFICADOS:,}",
     f"{IDENTIFICADOS / SNAPSHOTS * 100:.1f}%"),
    ("Confirmed by content diff (any state)", f"{CONFIRMADOS:,}",
     f"{CONFIRMADOS / SNAPSHOTS * 100:.1f}%"),
    ("Still unidentified", f"{SNAPSHOTS - IDENTIFICADOS:,}",
     f"{(SNAPSHOTS - IDENTIFICADOS) / SNAPSHOTS * 100:.1f}%"),
    ("Years covered", f"{por_anio.index.min()}\u2013{por_anio.index.max()}", ""),
]
Markdown("\n".join(
    ["| | | |", "|---|---:|---|"]
    + [f"| {etiqueta} | {valor} | {nota} |" for etiqueta, valor, nota in filas]
))


### How deep the history goes

[@fig-snapshots-per-year] plots the snapshots by the year of the reform that
produced them, and it is the clearest available picture of when Mexico's
federal statute book actually changed. Three things stand out. The first
half-century is thin: the Court's database reaches back to 1917 and beyond —
the oldest snapshot in the collection is the Commercial Code of 1889, older
than the *Diario Oficial* coverage this project is built on — but reforms
before the 1970s are recorded in ones and tens per year, partly because there
were fewer and partly because the Court's retrospective coverage of them is
incomplete. Volume then rises through the last quarter of the century and
settles, from the 1990s onward, into a regime of well over a hundred reforms a
year across the collection. The final year is partial: it covers only the
months already elapsed when the crawl ran.


In [ ]:
#| label: fig-snapshots-per-year
#| fig-cap: "Snapshots per year of publication. The last year, in a lighter shade, is partial — it ends when the crawl ran. Hover on a bar for the exact count."
ultimo = por_anio.index.max()
colores = [LIGHT_BLUE if anio == ultimo else BLUE for anio in por_anio.index]
fig = go.Figure(
    go.Bar(
        x=por_anio.index, y=por_anio.values,
        marker=dict(color=colores, cornerradius=2),
        hovertemplate="%{x}: %{y:,} snapshots<extra></extra>",
    )
)
style_plotly(fig, yaxis_title="Snapshots")
fig.update_layout(bargap=0.2, showlegend=False, hovermode="x unified")
fig.add_annotation(x=ultimo, y=por_anio.iloc[-1], text="partial year",
                   yshift=14, showarrow=False, font=dict(color=MUTED, size=11))
fig.show(config=PLOTLY_CONFIG)


### How much of it is identified, and what the second signal adds

[@fig-link-status] is the figure to read before using this collection for
anything. It breaks the 3,724 snapshots down by the state the title signal
left them in, and splits each state by whether the content signal confirmed a
provision. The stacked pair is the point: the `ambiguous` bar's confirmed
segment counts dates that the title signal could not resolve and the content
signal could — the concrete return on the more expensive computation.


In [ ]:
#| label: fig-link-status
#| fig-cap: "Snapshots by title-link state, split by whether the content diff confirmed a provision. The confirmed part of the `ambiguous` bar is what the second signal recovers on its own."
orden = ["linked", "ambiguous", "claimed", "none"]
etiquetas = {
    "linked": "linked<br/>(one candidate)",
    "ambiguous": "ambiguous<br/>(several candidates)",
    "claimed": "claimed<br/>(taken by another snapshot)",
    "none": "none<br/>(no candidate that day)",
}
confirmados = [POR_CONFIRMACION.get(f"{e}|confirmed", 0) for e in orden]
sin_confirmar = [POR_CONFIRMACION.get(f"{e}|unconfirmed", 0) for e in orden]
fig = go.Figure()
fig.add_bar(
    x=[etiquetas[e] for e in orden], y=confirmados, name="confirmed by content diff",
    marker=dict(color=BLUE), hovertemplate="%{y:,} snapshots<extra>confirmed</extra>",
)
fig.add_bar(
    x=[etiquetas[e] for e in orden], y=sin_confirmar, name="not confirmed",
    marker=dict(color=LIGHT_BLUE), hovertemplate="%{y:,} snapshots<extra>not confirmed</extra>",
)
style_plotly(fig, yaxis_title="Snapshots", height=440)
fig.update_layout(barmode="stack", bargap=0.35, hovermode="x unified",
                  legend=dict(orientation="h", y=1.12, x=0))
fig.update_yaxes(tickformat=",")
fig.show(config=PLOTLY_CONFIG)


In [ ]:
#| label: link-summary
#| code-summary: "The two sentences of the paragraph below, computed rather than written"
amb = ESTADOS.get("ambiguous", 0)
sin_ident = {int(a): n for a, n in resumen["unidentified_per_year"].items()}
SIN_IDENT_TOTAL = sum(sin_ident.values())
RECIENTES = sum(n for a, n in sin_ident.items() if a >= 2000)
ANTIGUOS = sum(n for a, n in sin_ident.items() if a < 1990)
Markdown(
    f"Read together, the two signals identify {IDENTIFICADOS:,} of the "
    f"{SNAPSHOTS:,} snapshots ({IDENTIFICADOS / SNAPSHOTS * 100:.1f}%): "
    f"{ENLAZADOS:,} where a single same-day provision named the instrument, plus "
    f"{RESCATADOS:,} of the {amb:,} ambiguous dates where the content diff picked "
    f"one candidate out of several. That leaves {SNAPSHOTS - IDENTIFICADOS:,} "
    f"snapshots ({(SNAPSHOTS - IDENTIFICADOS) / SNAPSHOTS * 100:.1f}%) whose "
    f"originating provision is still unknown, and — against intuition — these are not "
    f"mostly the oldest ones: {RECIENTES:,} of them ({RECIENTES / max(SIN_IDENT_TOTAL, 1) * 100:.0f}%) "
    f"fall in 2000 or later, and only {ANTIGUOS:,} before 1990. The reason is that the "
    f"unresolved cases are dominated by days on which the gazette published several "
    f"provisions all naming the same law — a modern habit, since a single reform "
    f"package now often arrives as several decrees — and the content diff was not "
    f"decisive between them. The genuinely old failures are of the other kind: "
    f"{ESTADOS.get('none', 0):,} snapshots have no same-day candidate at all. "
    f"Note also that confirmation is not the same as agreement: of the "
    f"{ENLAZADOS:,} title-linked snapshots, "
    f"{POR_CONFIRMACION.get('linked|confirmed', 0):,} were independently confirmed by "
    f"content, and the rest simply had no digital candidate text to check against."
)


### Instrument by instrument

[@tbl-instruments] lists the twenty-five instruments with the most snapshots,
which is a reasonable proxy for the most heavily reformed pieces of Mexican
federal law. It is the same information the collection's `MANIFEST.md` carries
— the file a human reads in full before publishing — reordered by size rather
than by confidence.

The table has four columns worth explaining. **Snapshots** is how many dated
versions of that instrument the collection holds, and therefore how many times
its text changed within the Court's coverage; the Constitution leads by a wide
margin, which is itself a substantive fact about Mexican constitutionalism.
**Span** gives the first and last year covered, which is where the Court's
retrospective reach shows: an instrument may be older than its span suggests.
**Title-linked** is the share of that instrument's snapshots that a single
same-day provision naming the law resolved on its own — the column to consult
before relying on a particular law's history, since coverage is far from
uniform. **Confirmed** counts the snapshots whose provision was corroborated
by comparing texts, the strongest evidence the pipeline produces; it is
reported as a count rather than a share because it can exceed the title-linked
count, precisely when it resolves dates the title left ambiguous. A low
title-linked figure never means the text is wrong — it means the collection
cannot yet say, from that signal alone, which publication produced that
version.


In [ ]:
#| label: tbl-instruments
#| tbl-cap: "The twenty-five instruments with the most snapshots. `Title-linked` is the share the title signal resolved on its own; `Confirmed`, the number of snapshots corroborated by comparing the candidate provision's own text."
TOPN = 25
tabla = instrumentos.head(TOPN)
lineas = [
    "| Instrument | Slug | Snapshots | Span | Title-linked | Confirmed |",
    "|---|---|---:|---|---:|---:|",
]
for fila in tabla.itertuples():
    span = (
        f"{fila.first_year}\u2013{fila.last_year}"
        if fila.first_year and fila.last_year and fila.first_year != fila.last_year
        else (str(fila.first_year) if fila.first_year else "\u2014")
    )
    nombre = fila.nombre if len(fila.nombre) <= 70 else fila.nombre[:67] + "\u2026"
    lineas.append(
        f"| {nombre} | `{fila.slug}` | {fila.snapshots:,} | {span} | "
        f"{fila.linked / fila.snapshots * 100:.0f}% | {fila.diff_confirmed:,} |"
    )
Markdown("\n".join(lineas))


In [ ]:
#| label: tail-summary
#| code-summary: "The distribution behind the table's long tail"
un_solo = int((instrumentos["snapshots"] == 1).sum())
todo_enlazado = int((instrumentos["snapshots"] == instrumentos["linked"]).sum())
sin_enlace = int((instrumentos["linked"] == 0).sum())
mediana = int(instrumentos["snapshots"].median())
Markdown(
    f"Beyond the top of the table the distribution is very uneven: the median "
    f"instrument has {mediana} snapshots, {un_solo} of the {CON_DIRECTORIO} "
    f"instruments have exactly one — laws never reformed since they were enacted, "
    f"or enacted so recently that the Court's table holds a single row — "
    f"{todo_enlazado} have every one of their snapshots resolved by the title signal "
    f"alone, and {sin_enlace} have none resolved by it at all."
)


## Cases the pipeline could not handle on its own

The crawl runs unattended for the great majority of the catalogue: 313 of the
316 entries have a directory built entirely by the general pipeline. Four
entries did not come for free — one of them (`lisipl`) only after the pipeline
itself was changed, two (`lfca`, `lfiiedb`) by scripts written for that single
law, and one (`oga`) not at all. Those four are the interesting
part, and they are
documented here rather than in a footnote, because this is where a reader can
judge how the collection was actually built. Each one below is a *pattern*, not
a curiosity, and each is recorded with the state it was left in.

### `lisipl` — a catalogue name no search can match

The Chamber lists one tax law under a name that carries a 255-character
parenthetical alternate title. Searched verbatim, it matches nothing at all in
the Court's database: full-text search does not degrade gracefully when the
query is a paragraph. Rather than making the crawler guess how to shorten a
name — a heuristic that would silently mis-identify other instruments — the
catalogue entry accepts an optional hand-written override, **`nombre_scjn`**,
naming the exact string to search for. Its whole entry in `catalogo.json` is
short enough to show in full:

```json
{
  "nombre": "IMPUESTO sobre Servicios Expresamente Declarados de Interés Público por Ley, en los que Intervengan Empresas Concesionarias de Bienes del Dominio Directo de la Nación (LEY que establece, reforma y adiciona las disposiciones relativas a diversos impuestos)",
  "abrev": "lisipl",
  "nombre_scjn": "LEY DEL IMPUESTO SOBRE SERVICIOS EXPRESAMENTE DECLARADOS DE INTERES PUBLICO POR LEY, EN LOS QUE INTERVENGAN EMPRESAS CONCESIONARIAS DE BIENES DEL DOMINIO DIRECTO DE LA NACION"
}
```

The two strings name the same law: what `nombre_scjn` drops is the
parenthetical `(LEY que establece, reforma y adiciona las disposiciones
relativas a diversos impuestos)` that the Chamber appends, and what it keeps is
the law's own title as the Court spells it. `nombre_a_buscar` prefers it over
`nombre` when it is present, so it is `nombre_scjn` — not the Chamber's name —
that ends up in the snapshot's `nombre_buscado` header. This is the
one-source's-name-against-the-other's pairing of the header block, with the
Chamber's half replaced by hand precisely because it was unusable as a query;
the low similarity score that results is correct, and is why `lisipl` is one of
the three instruments the confidence check flags. A catalogue refresh carries
the override forward rather than overwriting it (`merge_catalog_overrides`),
so re-seeding from the Chamber never silently loses it. One entry uses it
today. The general rule this encodes is worth stating: where an automated
guess would be unverifiable, the pipeline prefers an explicit, versioned,
human decision.

### `lfca` — a new law that repeals an indexed one

The Federal Cinema and Audiovisual Law was published in May 2026 and repealed
the 1992 Federal Cinematography Law. The Court has no entry for the new law;
what it does have is the repealed one, with its entire reform history. So the
corpus was built in two halves: the **history**, crawled under the repealed
law's name, and the **current text**, taken directly from the *Diario Oficial*
(provision `codNota` 5788357, 22 May 2026) and converted with `nota2md`. One
detail is worth recording: the Court's own last row for the repealed law is
dated the day the new law appeared and announces its enactment, but its body is
still the old 1992 text — so that row was discarded, and 22 May 2026 has
exactly one snapshot, the gazette's.

This is where the header's `fuente` field earns its keep. Nine of this
instrument's ten snapshots read `fuente: scjn` and one reads `fuente: dof`, so
the two halves are distinguishable without opening a single file, and the same
field appears per entry in the instrument's `indice.json`. Consumers must treat
that index field as **optional**: the normal pipeline never writes it, and its
absence means `scjn`.

### `lfiiedb` — the same pattern, without the history

The Law for Promoting Investment in Strategic Infrastructure for Development
with Well-being was published in April 2026 and has never been reformed. The
Court does not index it, and there is no repealed predecessor to borrow a
history from. Its corpus is therefore a single file taken from the gazette
(`codNota` 5784517), with a link that is known by construction rather than
inferred. It is the minimal case of the previous pattern.

### `oga` — outside the gazette's reach

The General Ordinance of the Navy is the one catalogue entry with no directory
at all. It was published between 1 and 8 January **1912**, before the *Diario
Oficial* as this project covers it (1917) begins, and the Court does not index
it either. The two provisions its Chamber history lists are not the ordinance
but later instruments that reformed it. The gap is real, it is understood, and
it is deliberately left open — the alternative would be to fabricate coverage
the sources do not support.

### What these cases have in common


In [ ]:
#| label: casos-summary
#| code-summary: "The flagged-confidence counts cited below"
motivos = {}
for datos in resumen["confidence"].values():
    motivos[datos["motivo"]] = motivos.get(datos["motivo"], 0) + 1
Markdown(
    f"A side effect worth knowing about: the hand-built cases are exactly the "
    f"ones the automated confidence check flags. Of the {CON_DIRECTORIO} "
    f"instruments with a directory, {motivos.get('confiable', 0)} are classified "
    f"`confiable`, {motivos.get('sospechoso', 0)} `sospechoso` and "
    f"{motivos.get('bajo_umbral', 0)} below the similarity threshold. The flagged "
    f"ones are `lfca` and `lfiiedb` — whose stored title is the gazette's decree "
    f"wording rather than the catalogue's law name, so a low similarity score is "
    f"correct and expected — and `lisipl`, whose catalogue name carries a "
    f"250-character alternate title that the Court's search never matches and which "
    f"is therefore searched under a hand-written override. Three flags, three "
    f"explanations, no unexplained ones: that is the state the collection is in, and "
    f"the check is worth keeping precisely because it is loud."
)


Two of the three resolved cases are the same situation — a recent law the
Court has not indexed yet, closed by taking the text from the gazette — and
both are **reversible**. When the Court indexes the law, the hand-built
directory should be reviewed and replaced by an ordinary crawl, and the
single-purpose script that built it retired. The refresh logic already
cooperates: an instrument with no snapshots is retried on every run, so the
only manual act left is deleting the hand-built directory once the normal path
works.

[@fig-decision] states the pattern as a rule, so the next law of this shape is
handled by consulting a diagram instead of improvising.


::: {#fig-decision}

```{mermaid}
flowchart LR
    Q{SCJN indexes<br/>the law?} -->|yes| N[normal crawl]
    Q -->|no| H{Does it abrogate<br/>an indexed law?}
    H -->|yes| B["history: crawl the abrogated law<br/>current text: DOF codNota"]
    H -->|no| O{Published<br/>after 1917?}
    O -->|yes| S[single DOF snapshot]
    O -->|no| X[out of scope]
```

The rule for an instrument the Court's database does not have. `lfca` takes
the middle branch, `lfiiedb` the single-snapshot branch, `oga` the last one.

:::


## How this collection is published

The corpus is published, since August 2026, as the
[`scjn-leyes`](https://github.com/INGEOTEC/LegalIA/releases/tag/scjn-leyes)
GitHub release.

**One reproducible tarball per instrument.** The packaging step writes one
`<slug>.tgz` for each instrument rather than a single archive for the whole
collection, because a consumer almost always wants one law, and a later run
then has to re-upload only the laws that changed. Every member is prefixed
with the instrument's slug, so a tarball unpacks anywhere without stepping on
anything:

```
<slug>/<date>.md            the snapshots, each with its provenance header
<slug>/indice.json          the codNota link and the signals behind it
<slug>/notas/nota-<cod>.md  the DOF text of every candidate considered
```

The gazette notes travel *with* the snapshots on purpose: they are the text
each link was decided against, so shipping both makes a link auditable without
going back to the network. Two small text assets accompany the tarballs — a
`MANIFEST.md` ranking every instrument by confidence, and a `SHA256SUMS.txt`.
Each tarball is **byte-reproducible**: gzip is stamped with a zero timestamp,
members are added in sorted order, and their times, modes and ownership are
fixed. Identical data therefore yields an identical file, which is what makes
it possible to tell an unchanged instrument from a changed one by comparing
bytes rather than guessing — the same property the `historial-legislativo`
release relies on.

**Published as GitHub release assets**, like the DOF notes archive and the
reform-history collection, and for the same reason: data is never committed to
this repository.

**Published by hand, always, by design.** This is the one operational
difference between this collection and the others, and it is a statement about
the source rather than a gap in automation. The notes archive is rebuilt and
republished monthly by a workflow with no human in the loop, because its source
is the gazette's own service and an error there is the gazette's error. The
Court's search, by contrast, can return a completely wrong document, and a
wrong document would enter the corpus looking exactly like a right one — so
the packaging script never invokes `gh`, no workflow may publish this
collection, and `MANIFEST.md` is read in full before anything is uploaded.
Separating packaging from publishing keeps the human judgement where the risk
actually is.

**Reading it back.** Because one instrument is one asset, downloading a law
needs no library at all:

```bash
gh release download scjn-leyes --repo INGEOTEC/LegalIA --pattern 'lft.tgz'
tar xzf lft.tgz          # -> lft/<date>.md, lft/indice.json, lft/notas/
```

A download function in the same family as the ones already serving the other
collections — so that reading the corpus from Python is a call rather than an
exercise — is still to be written.

**Licensing and attribution are still open.** The snapshots are the Supreme
Court's editorial work over public legal texts, and the terms on which they may
be redistributed have yet to be settled; this page is where the answer will be
recorded. Nothing about the pipeline depends on that answer — the scripts
reconstruct the corpus from the sources on any machine — but what the release
carries does.


## Methodological note {.unnumbered .appendix}

The material described here is built by four scripts in
[`scripts/`](https://github.com/INGEOTEC/LegalIA/tree/master/scripts) —
`extract_scjn_titles.py`, `fetch_scjn_legislacion.py`,
`enlaza_scjn_legislacion.py` and `empaqueta_scjn_leyes.py` — plus two
single-law scripts for the hand-built cases, `construye_lfca.py` and
`fetch_lfiiedb_dof.py`. The SCJN-specific logic they share lives in
`nota2md.scjn`; the *Diario Oficial* side comes from
[`dofjson`](https://pypi.org/project/dofjson/) and
[`nota2md`](https://pypi.org/project/nota2md/), both on PyPI.

The corpus is published as release assets and is never committed to git, so
this page does not read it directly — downloading 315 tarballs to draw two
figures would be a poor trade. It reads
[`data/scjn-leyes-summary.json`](https://github.com/INGEOTEC/LegalIA/blob/master/website/pages/data/scjn-leyes-summary.json),
an aggregate summary produced by `scripts/resume_scjn_leyes.py` from the crawl
that the August 2026 release was packaged from: headline counts, snapshots per
year, the title-link states crossed with content confirmation, and one row per
instrument. That script reuses the very function `empaqueta_scjn_leyes.py`
uses to build `MANIFEST.md`, so the numbers on this page, the numbers a human
reads before publishing, and the manifest shipped with the release cannot
drift apart. Regenerating the summary after a fresh crawl is one command:

```bash
./scripts/resume_scjn_leyes.py --outdir scripts/scjn \
    --destino website/pages/data/scjn-leyes-summary.json
```

This page was written by the LegalIA team together with Claude, Anthropic's
coding assistant, through [Claude Code](https://claude.com/claude-code): the
assistant implemented the summary step, produced the figures and drafted the
accompanying text. The authors verified the numbers against the corpus on disk
and are responsible for the interpretations advanced here.
